# MCP vs tool_use: Architecture & When to Use Each

## Table of contents
- [What is tool_use?](#what-is-tool_use)
- [What is MCP?](#what-is-mcp)
- [Architecture comparison](#architecture-comparison)
- [Setup](#setup)
- [Part 1: Raw tool_use demo](#part-1-raw-tool_use-demo)
- [Part 2: MCP as a standardization layer](#part-2-mcp-as-a-standardization-layer)
- [Decision table: when to use each](#decision-table-when-to-use-each)
- [quick-reference](#quick-reference)

## What is tool_use?

`tool_use` is the **Claude API mechanism** for invoking external functions.

When you provide Claude with a list of tool schemas, it reads the `description`
and `input_schema` of each tool, then outputs a `tool_use` block — a structured
JSON request — instead of a plain text reply. Your code executes the function
and returns a `tool_result` block.

```
User message
    |
Claude reads tool schemas (name + description + input_schema)
    | decides which tool to call
tool_use block { name, id, input }
    |
Your code executes the function
    |
tool_result block { tool_use_id, content }
    |
Claude produces final answer
```

**Key characteristics:**
- Tools defined in your **client code** alongside the API call
- Claude never executes code — it only outputs structured requests
- Tools are scoped to a single `messages.create` call (no persistence)
- Ideal for **application-specific, bespoke integrations**

## What is MCP?

**Model Context Protocol (MCP)** is a **standardized protocol layer** that defines
how AI models discover and interact with external tools, data, and workflows.

MCP sits *above* the `tool_use` API mechanism. MCP Tools still use `tool_use`/`tool_result`
at the API level — but MCP adds a standardization layer on top:

| What MCP adds | Benefit |
|---|---|
| **Three primitives** (Tools, Resources, Prompts) | Structured contract for different interaction types |
| **Server-side registration** | Capabilities defined in a server, not client code |
| **Discovery protocol** | Client asks server: "what do you expose?" automatically |
| **Transport layer** (stdio / SSE) | Standardized communication channel |
| **Server lifecycle** | Connect, initialize, disconnect |

### The three MCP primitives

| Primitive | Controller | Side effects? | Use for |
|---|---|---|---|
| **Tools** | Model (Claude decides when to call) | Yes | Write DB, call API, execute code |
| **Resources** | Application (host injects as context) | No | Read files, query data |
| **Prompts** | User (manually selects from menu) | No | Reusable workflow templates |

## Architecture comparison

```
+-----------------------------------------------+
|               RAW tool_use                    |
+-----------------------------------------------+
|                                               |
|  Your Application        Claude API           |
|  +-----------------+     +---------------+   |
|  | tools: [{       |     |               |   |
|  |   name,         | --> | reads schemas |   |
|  |   description,  |     | outputs       |   |
|  |   input_schema  | <-- | tool_use      |   |
|  | }]              |     | block         |   |
|  | handler()       | --> | tool_result   |   |
|  +-----------------+     +---------------+   |
|                                               |
+-----------------------------------------------+

+------------------------------------------------+
|                MCP Protocol                   |
+------------------------------------------------+
|                                               |
|  MCP Client         MCP Server   Claude API   |
|  +------------+     +--------+  +-----------+ |
|  |            | --> |exposes:|  |           | |
|  | Claude SDK |     | Tools  |  | same      | |
|  | + MCP      | <-- | Resour.|  | tool_use/ | |
|  | client     |     | Prompts|  | tool_result| |
|  |            | --> |schemas|--> protocol  | |
|  +------------+     +--------+  +-----------+ |
|                                               |
+------------------------------------------------+
```

> **Key insight**: MCP Tools still use `tool_use`/`tool_result` at the Claude API level.
> MCP is the standardization layer **above** the API, not a replacement for it.

## Setup

Ensure `@anthropic-ai/sdk` is installed and `ANTHROPIC_API_KEY` is in your `.env` file.

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });

console.log('Client ready. Model:', MODEL_NAME);

## Part 1: Raw tool_use demo

This is the standard approach when calling the Claude API directly.
You define tools in your client code, run the agentic loop yourself, and
handle tool execution.

The example below uses a `get_stock_price` tool to demonstrate the complete
`tool_use` → execute → `tool_result` → final answer flow.

In [ ]:
// Tool schema — what Claude sees
const stockTool: Anthropic.Tool = {
  name: 'get_stock_price',
  description:
    'Get the current price of a stock by its ticker symbol. ' +
    'Use this when the user asks about stock prices or market values.',
  input_schema: {
    type: 'object',
    properties: {
      ticker: {
        type: 'string',
        description: 'The stock ticker symbol, e.g. "AAPL" or "GOOGL"',
      },
    },
    required: ['ticker'],
  },
};

// Handler — your code that runs when Claude requests the tool
function getStockPrice(ticker: string): { ticker: string; price: number; currency: string } {
  const prices: Record<string, number> = {
    AAPL: 189.3,
    GOOGL: 175.8,
    MSFT: 415.2,
    AMZN: 198.6,
  };
  return { ticker, price: prices[ticker] ?? 100.0, currency: 'USD' };
}

console.log('Tool schema:', stockTool.name);
console.log('Handler test:', getStockPrice('AAPL'));

In [ ]:
// Standard agentic loop for raw tool_use
async function askWithTool(question: string): Promise<string> {
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: question }];

  while (true) {
    const response = await client.messages.create({
      model: MODEL_NAME,
      max_tokens: 512,
      tools: [stockTool],
      messages,
    });

    console.log('  stop_reason:', response.stop_reason);

    if (response.stop_reason === 'end_turn') {
      const text = response.content.find(b => b.type === 'text');
      return text?.type === 'text' ? text.text : '(no text)';
    }

    messages.push({ role: 'assistant', content: response.content });

    const toolResults: Anthropic.ToolResultBlockParam[] = [];
    for (const block of response.content) {
      if (block.type !== 'tool_use') continue;
      console.log('  -> tool_use:', block.name, JSON.stringify(block.input));
      const result = getStockPrice((block.input as { ticker: string }).ticker);
      console.log('  <- tool_result:', result);
      toolResults.push({
        type: 'tool_result',
        tool_use_id: block.id,
        content: JSON.stringify(result),
      });
    }
    messages.push({ role: 'user', content: toolResults });
  }
}

const answer = await askWithTool("What's the current price of Apple stock?");
console.log('\nFinal answer:', answer);

## Part 2: MCP as a standardization layer

In the MCP world, tools are defined **inside an MCP server** — not in your client code.
The MCP client (your application) connects to the server, **discovers** what
tools/resources/prompts it exposes, and uses them through a standardized protocol.

The TypeScript interfaces below represent MCP primitive definitions as the
`@modelcontextprotocol/sdk` library models them.

> When an MCP Tool is called, it still produces `tool_use`/`tool_result` blocks
> at the Claude API level. MCP is a wrapper — not a different API.

In [ ]:
// TypeScript types representing MCP primitives
// (mirrors @modelcontextprotocol/sdk type definitions)

// MCP Tool — model-controlled, has side effects
interface McpToolDefinition {
  name: string;
  description: string;
  inputSchema: {
    type: 'object';
    properties: Record<string, { type: string; description: string; enum?: string[] }>;
    required: string[];
  };
}

// MCP Resource — application-controlled, read-only, no side effects
interface McpResourceDefinition {
  uri: string;       // e.g. 'file:///data/report.csv' or 'db://users/42'
  name: string;
  description: string;
  mimeType: string;  // e.g. 'text/csv', 'application/json'
}

// MCP Prompt — user-controlled, reusable workflow template
interface McpPromptDefinition {
  name: string;
  description: string;
  arguments: Array<{
    name: string;
    description: string;
    required: boolean;
  }>;
}

console.log('MCP primitive types defined: Tool | Resource | Prompt');

In [ ]:
// Same stock price capability expressed as all three MCP primitives
// to illustrate the difference between them.

// As a Tool: Claude CALLS it to get price data (side effect: external API call)
const mcpStockTool: McpToolDefinition = {
  name: 'get_stock_price',
  description: 'Get the current price of a stock. Use when the user asks about stock prices.',
  inputSchema: {
    type: 'object',
    properties: {
      ticker: { type: 'string', description: 'Stock ticker symbol, e.g. "AAPL"' },
    },
    required: ['ticker'],
  },
};

// As a Resource: read-only snapshot of portfolio data — no API call, pure data
const mcpPortfolioResource: McpResourceDefinition = {
  uri: 'file:///data/portfolio.csv',
  name: 'portfolio-snapshot',
  description: 'Current portfolio holdings as of market close',
  mimeType: 'text/csv',
};

// As a Prompt: user-triggered analysis workflow template
const mcpAnalysisPrompt: McpPromptDefinition = {
  name: 'portfolio-analysis',
  description: 'Standard workflow for analyzing portfolio performance',
  arguments: [
    { name: 'period', description: 'Time period, e.g. "Q1 2026"', required: true },
    { name: 'focus', description: 'Analysis focus area', required: false },
  ],
};

// MCP clients convert Tool definitions to Claude tool_use schemas automatically.
// Resources and Prompts are handled differently (injected as context, not tool_use calls).
function mcpToolToClaudeSchema(mcpTool: McpToolDefinition): Anthropic.Tool {
  return {
    name: mcpTool.name,
    description: mcpTool.description,
    input_schema: {
      type: 'object',
      properties: mcpTool.inputSchema.properties,
      required: mcpTool.inputSchema.required,
    },
  };
}

const claudeSchema = mcpToolToClaudeSchema(mcpStockTool);

console.log('=== MCP Tool -> Claude tool_use schema (same structure) ===');
console.log(JSON.stringify(claudeSchema, null, 2));
console.log('\n=== MCP Resource (read-only, injected as context) ===');
console.log(JSON.stringify(mcpPortfolioResource, null, 2));
console.log('\n=== MCP Prompt (user-triggered template) ===');
console.log(JSON.stringify(mcpAnalysisPrompt, null, 2));

## Decision table: when to use each

| Scenario | Approach |
|---|---|
| App-specific tools tightly coupled to your codebase | raw tool_use |
| Quick prototype — minimal setup needed | raw tool_use |
| Tools that change dynamically per request | raw tool_use |
| Standardized tools reused across multiple AI apps | MCP |
| Read-only data (files, DB reads, API reads) | MCP Resource |
| User-selectable workflow templates | MCP Prompt |
| Production deployment with multiple clients | MCP |
| Remote server (cloud, multi-user) | MCP + SSE transport |
| Local tools (same machine, single user) | MCP + stdio transport |

### Exam rule

> Scenario mentions **"standardize", "multiple apps", "reuse", "remote",
> "discovery"** or **"three primitive types"** → the answer involves **MCP**.
>
> Scenario is a direct API call in a single application → **raw tool_use**.

In [ ]:
// Exam Practice — classify each scenario
type Approach = 'raw tool_use' | 'MCP Tool' | 'MCP Resource' | 'MCP Prompt';

interface Scenario {
  description: string;
  answer: Approach;
  reason: string;
}

const scenarios: Scenario[] = [
  {
    description: 'Claude must write analysis results to a database',
    answer: 'MCP Tool',
    reason: 'Write operation with side effects, model-controlled -> Tool',
  },
  {
    description: 'Claude must read the contents of a CSV file',
    answer: 'MCP Resource',
    reason: 'Read-only data, no side effects, app-controlled -> Resource',
  },
  {
    description: 'Analysts trigger a "Quarterly Report Analysis" workflow from the UI',
    answer: 'MCP Prompt',
    reason: 'User-selected reusable workflow template -> Prompt',
  },
  {
    description: 'A single chatbot calls a weather API only for its own use',
    answer: 'raw tool_use',
    reason: 'App-specific, not shared across systems, bespoke -> raw tool_use',
  },
  {
    description: 'Three different AI products share access to the same CRM integration',
    answer: 'MCP Tool',
    reason: 'Cross-app standardization = MCP; CRM writes have side effects = Tool',
  },
];

console.log('=== Primitive Selection Practice ===\n');
for (const s of scenarios) {
  console.log(`Scenario : ${s.description}`);
  console.log(`Answer   : ${s.answer}`);
  console.log(`Reason   : ${s.reason}`);
  console.log('-'.repeat(70));
}

## Quick-Reference

### What MCP adds over raw tool_use

| Feature | Raw tool_use | MCP |
|---|---|---|
| Tools (function calling) | Yes | Yes |
| Resources (read-only data) | No | Yes |
| Prompts (user templates) | No | Yes |
| Tool discovery | Manual (hardcoded) | Automatic (server exposes) |
| Transport layer | N/A | stdio / SSE |
| Multi-client reuse | Difficult | Yes, by design |

### Three-primitive judgment (exam high-frequency)

```
Has side effects (write/execute)?   -> Tool     (model-controlled)
Read-only data?                     -> Resource  (app-controlled)
User-selected workflow?             -> Prompt   (user-controlled)
```

### Transport judgment

```
Local / same machine?               -> stdio
Remote / cloud / multi-client?      -> SSE (Streamable HTTP)
```

### Next notebooks

- `01_mcp_primitives.ipynb` — deep dive into Tools, Resources, Prompts with hands-on TypeScript
- `02_mcp_server_basics.ipynb` — build a working MCP server with all three primitives
- `03_tool_schema_design.ipynb` — spot and fix broken tool schemas (high-frequency exam pattern)
- `04_mcp_client_with_claude.ipynb` — end-to-end: Claude + MCP client + error handling